In [ ]:
## Notebook 03 — Sentinel Treatment & Feature Engineering

**Input:** accepted_dtypedchanged_coldrop.csv — 2,260,701 rows × 28 columns  
**Output:** clean_pd_dataset.parquet — 2,260,701 rows × 73 columns  
**Purpose:** Re-apply dtypes (lost during CSV save in NB02), treat sentinel and 
extreme values, engineer features, and produce a parquet file for splitting and modeling.

---

### Note on Dtype Re-application
Due to the pyarrow version conflict in Notebook 02, dtypes were lost when the 
intermediate file was saved as CSV. This notebook reloads that CSV and re-applies 
all dtype conversions (Int64, category, datetime64) before proceeding with 
sentinel treatment. This is not redundant — it is a necessary pipeline step 
caused by the CSV serialization limitation documented in Notebook 02.

---

### Features Engineered

| Feature | Construction | Source columns dropped? |
|---|---|---|
| `credit_history_years` | (issue_d − earliest_cr_line) / 365.25 | Yes — both dropped after engineering |
| `fico_mean` | (fico_range_low + fico_range_high) / 2, rounded to Int64 | Source columns retained for now but drop before modelling |

---

### Sentinel & Extreme Value Treatment

| Column | Sentinel Rule | Sentinel → NaN? | Extreme Rule | Cap Applied |
|---|---|---|---|---|
| `annual_inc` | == 0 or < 2,000 | Yes | > 270,000 (99th pct) | Yes, at 270k |
| `dti` | < 0 or > 200 | Yes | > 43 | Yes, at 43 |
| `revol_util` | > 2.0 | Yes | 1.0–2.0 | Yes, capped at 1.0 |
| `credit_history_years` | > 80 years | Yes | < p1 (3.75 yrs) | Yes, at p1 |
| `int_rate` | None | — | > p99 (0.2677) | Yes |
| `installment` | None | — | < p1 or > p99 | Yes, winsorized |
| `loan_amnt` / `funded_amnt` | None | — | < p1 | Yes, lower tail only |
| `pub_rec` | > 40 | No (flagged only) | > p99 (2.0) | No |
| `open_acc`, `total_acc`, `inq_last_6mths`, `delinq_2yrs` | Threshold-based | Flagged only | > p99 | No |

---

### Categorical Cleaning

All categorical columns processed through a unified cleaning pipeline:
- Missing-like strings ("nan", "none", "n/a", "", "unknown") → replaced with "Unknown" category
- True NaN values → filled with "Unknown" (except `verification_status` — 33 true NaNs 
  retained here; imputed with train mode in Notebook 04)
- Rare labels (< 0.5% frequency) consolidated into "Other" category
- All columns confirmed as category dtype after processing

**purpose column:** 139,473 rows (6.17%) mapped to "Unknown". This is a known 
data quality characteristic of the dataset — purpose was not always captured at 
origination. "Unknown" is retained as a valid category and treated as a signal 
in its own right, as borrowers with unknown purpose may represent a distinct 
risk segment.

---

### Flag Columns — 45 total
Binary indicator columns created with suffix pattern `_was_sentinel`, 
`_was_missing`, `_was_extreme`. All unified to int8 dtype.

**Known limitation:** For `annual_inc` and `credit_history_years`, sentinel 
values were converted to NaN before missing flags were computed. This caused 
sentinel and missing flags to be identical for these columns. Both flags are 
retained. Materiality assessment confirmed zero impact on model outputs. 
Documented as a known limitation — see Model Documentation for full assessment.

---

### Output
- Shape: 2,260,701 × 73 columns
- Parquet saved successfully using pyarrow engine
- Dtypes fully preserved (45 × int8, 11 × Int64, 8 × float64, 8 × category, 1 × object)
- Remaining NaNs exist in modeling features — all imputed in Notebook 04 
  after train/val/test split using train-derived statistics only

In [4]:
import pandas as pd
df = pd.read_csv("/Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_dtypedchanged_coldrop.csv")
df.info()
df[['int_rate','revol_util']].head(10)

/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipykernel_13468/3019046353.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_dtypedchanged_coldrop.csv")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 28 columns):
 #   Column               Dtype  
---  ------               -----  
 0   id                   object 
 1   loan_amnt            float64
 2   funded_amnt          float64
 3   int_rate             float64
 4   installment          float64
 5   grade                object 
 6   sub_grade            object 
 7   home_ownership       object 
 8   annual_inc           float64
 9   verification_status  object 
 10  issue_d              object 
 11  loan_status          object 
 12  purpose              object 
 13  addr_state           object 
 14  dti                  float64
 15  delinq_2yrs          float64
 16  earliest_cr_line     object 
 17  fico_range_low       float64
 18  fico_range_high      float64
 19  inq_last_6mths       float64
 20  open_acc             float64
 21  pub_rec              float64
 22  revol_util           float64
 23  total_acc            float64
 24

,int_rate,revol_util
0,0.1399,0.297
1,0.1199,0.192
2,0.1078,0.562
3,0.1485,0.116
4,0.2245,0.645
5,0.1344,0.684
6,0.0917,0.845
7,0.0849,0.057
8,0.0649,0.345
9,0.1148,0.391


In [5]:
import pandas as pd
import numpy as np

# ====== INPUT ======
INPUT_PATH = "/Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_dtypedchanged_coldrop.csv"

# ====== LOAD ======
print("Loading data...")
df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns.\n")



# ====== Convert integer-like columns to Int64 ======
int_cols = [
    "delinq_2yrs", "fico_range_low", "fico_range_high", "inq_last_6mths",
    "open_acc", "pub_rec", "total_acc", "term_months", "emp_length_yrs", "default_flag"
]
for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
        print(f"Converted {col} → Int64")

# ====== Convert categorical columns ======
cat_cols = [
    "grade", "sub_grade", "home_ownership", "verification_status",
    "purpose", "addr_state", "application_type", "loan_status"
]
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype("category")
        print(f"Converted {col} → category")

# ======  Convert date columns ======
date_cols = ["issue_d", "earliest_cr_line"]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", format="%Y-%m-%d")
        print(f"Converted {col} → datetime64[ns]")

# ======  Quick sanity check ======
print("\n Conversion complete.\n")
print(df.dtypes)
print("\nSample of date columns:")
print(df[["earliest_cr_line", "issue_d"]].head(10))


Loading data...
Loaded 2,260,701 rows and 28 columns.

Converted delinq_2yrs → Int64
Converted fico_range_low → Int64
Converted fico_range_high → Int64
Converted inq_last_6mths → Int64
Converted open_acc → Int64
Converted pub_rec → Int64
Converted total_acc → Int64
Converted term_months → Int64
Converted emp_length_yrs → Int64
Converted default_flag → Int64
Converted grade → category
Converted sub_grade → category
Converted home_ownership → category
Converted verification_status → category
Converted purpose → category
Converted addr_state → category
Converted application_type → category
Converted loan_status → category
Converted issue_d → datetime64[ns]
Converted earliest_cr_line → datetime64[ns]

 Conversion complete.

id                             object
loan_amnt                     float64
funded_amnt                   float64
int_rate                      float64
installment                   float64
grade                        category
sub_grade                    category
home

In [6]:
# create credit_history_years without altering datetime columns
df["credit_history_years"] = (df["issue_d"] - df["earliest_cr_line"]).dt.days / 365.25

# Ensure dtype is float64 
df["credit_history_years"] = df["credit_history_years"].astype("float64")

print("✅ credit_history_years created successfully.")
print(df["credit_history_years"].head(10))
print("\nDtype of credit_history_years:", df["credit_history_years"].dtype)


✅ credit_history_years created successfully.
0    12.334018
1    16.000000
2    15.331964
3     7.247091
4    17.500342
5    28.167009
6    25.500342
7    16.829569
8    13.667351
9    21.081451
Name: credit_history_years, dtype: float64

Dtype of credit_history_years: float64


In [7]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 29 columns):
 #   Column                Dtype         
---  ------                -----         
 0   id                    object        
 1   loan_amnt             float64       
 2   funded_amnt           float64       
 3   int_rate              float64       
 4   installment           float64       
 5   grade                 category      
 6   sub_grade             category      
 7   home_ownership        category      
 8   annual_inc            float64       
 9   verification_status   category      
 10  issue_d               datetime64[ns]
 11  loan_status           category      
 12  purpose               category      
 13  addr_state            category      
 14  dti                   float64       
 15  delinq_2yrs           Int64         
 16  earliest_cr_line      datetime64[ns]
 17  fico_range_low        Int64         
 18  fico_range_high       Int64         
 19  

In [10]:
# === Quick stats ===
num_cols = [
    'loan_amnt', 'funded_amnt', 'int_rate', 'installment',
    'annual_inc', 'dti', 'delinq_2yrs', 'inq_last_6mths',
    'open_acc', 'pub_rec', 'total_acc', 'fico_range_low',
    'fico_range_high', 'revol_util', 'credit_history_years',
    'term_months', 'emp_length_yrs'
]

# summary statistics
print("\n=== Summary Statistics (with 1st, 50th, 99th percentiles) ===")
print(df[num_cols].describe(percentiles=[0.01, 0.5, 0.99]).T)

# value_counts for potential sentinel columns
suspect_cols = ['annual_inc', 'dti', 'int_rate', 'revol_util', 'fico_range_low', 'fico_range_high']
for col in suspect_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(20))



=== Summary Statistics (with 1st, 50th, 99th percentiles) ===
                          count          mean            std       min  \
loan_amnt             2260668.0  15046.931228    9190.245488     500.0   
funded_amnt           2260668.0  15041.664057    9188.413022     500.0   
int_rate              2260668.0      0.130928       0.048321    0.0531   
installment           2260668.0    445.806823     267.173535      4.93   
annual_inc            2260664.0  77992.428687  112696.199574       0.0   
dti                   2258957.0     18.824196      14.183329      -1.0   
delinq_2yrs           2260639.0      0.306879        0.86723       0.0   
inq_last_6mths        2260638.0      0.576835       0.885963       0.0   
open_acc              2260639.0     11.612402       5.640861       0.0   
pub_rec               2260639.0      0.197528       0.570515       0.0   
total_acc             2260639.0     24.162552      11.987528       1.0   
fico_range_low        2260668.0    698.588205    

In [11]:
# Count of zeros and low incomes
low_income_count = (df['annual_inc'] < 10_000).sum()
zero_income_count = (df['annual_inc'] == 0).sum()

# Count of very large incomes
high_income_count = (df['annual_inc'] > 1_000_000).sum()

print("Low (<10k):", low_income_count)
print("Zero income:", zero_income_count)
print("High (>1M):", high_income_count)


Low (<10k): 4789
Zero income: 1667
High (>1M): 583


In [12]:
df.loc[df['annual_inc'] > 1_000_000, 'annual_inc'].value_counts().head(10)

annual_inc
1200000.0    58
1100000.0    42
1500000.0    32
2000000.0    18
1400000.0    18
1250000.0    18
1300000.0    15
3000000.0    12
1050000.0     9
1800000.0     7
Name: count, dtype: int64

In [13]:
import numpy as np
import pandas as pd

col = "annual_inc"

if col not in df.columns:
    raise KeyError(f"{col} not found in df")

# 1) Sentinel flag: annual_inc == 0 -> sentinel
#    (We treat 0 as sentinel here.)
df["annual_inc_was_sentinel"] = ((df[col] == 0) & df[col].notna()).astype("int8")

# Convert sentinel entries to NaN (so they are treated as missing from now on)
df.loc[df["annual_inc_was_sentinel"] == 1, col] = np.nan

# 2) Missing flag (after sentinel conversion)
df["annual_inc_was_missing"] = df[col].isna().astype("int8")

# 3) Extreme flag: mark values > 1,000,000 as extreme (1/0)
df["annual_inc_was_extreme"] = (df[col] > 1_000_000).astype("int8")

# Diagnostics before capping
total_rows = len(df)
sentinel_count = int(df["annual_inc_was_sentinel"].sum())
missing_count = int(df["annual_inc_was_missing"].sum())
extreme_count = int(df["annual_inc_was_extreme"].sum())

print(">>> annual_inc diagnostics BEFORE capping")
print(f"Total rows: {total_rows:,}")
print(f"Sentinel (==0) converted to NaN: {sentinel_count:,} ({sentinel_count/total_rows:.4%})")
print(f"Missing (NaN) now: {missing_count:,} ({missing_count/total_rows:.4%})")
print(f"Extreme (>1M) flagged: {extreme_count:,} ({extreme_count/total_rows:.4%})")

# Show top extreme values 
if extreme_count > 0:
    print("\nTop extreme values ( > 1M ) by frequency:")
    print(df.loc[df[col] > 1_000_000, col].value_counts().head(20))

# 4) Cap (winsorize) to 270k
cap_value = 270_000.0
# Only cap non-missing values
df[col] = df[col].where(df[col].isna(), np.minimum(df[col], cap_value))

# Diagnostics after capping
post_max = df[col].max(skipna=True)
num_capped = int(((df[col] == cap_value)).sum())
print("\n>>> annual_inc diagnostics AFTER capping")
print(f"Number of rows set to cap ({cap_value:,}): {num_capped:,}")
print(f"Max after capping: {post_max:,}")

# Quick distribution snapshot
print("\nannual_inc percentiles after capping:")
print(df[col].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))



>>> annual_inc diagnostics BEFORE capping
Total rows: 2,260,701
Sentinel (==0) converted to NaN: 1,667 (0.0737%)
Missing (NaN) now: 1,704 (0.0754%)
Extreme (>1M) flagged: 583 (0.0258%)

Top extreme values ( > 1M ) by frequency:
annual_inc
1200000.0    58
1100000.0    42
1500000.0    32
2000000.0    18
1400000.0    18
1250000.0    18
1300000.0    15
3000000.0    12
1050000.0     9
1800000.0     7
1020000.0     7
1140000.0     6
6000000.0     6
2300000.0     6
7000000.0     5
1150000.0     5
1750000.0     4
7500000.0     4
1900000.0     4
1450000.0     4
Name: count, dtype: int64

>>> annual_inc diagnostics AFTER capping
Number of rows set to cap (270,000.0): 23,086
Max after capping: 270,000.0

annual_inc percentiles after capping:
count    2.258997e+06
mean     7.621886e+04
std      4.458086e+04
min      3.600000e-01
1%       1.700000e+04
5%       2.800000e+04
50%      6.500000e+04
95%      1.600000e+05
99%      2.700000e+05
max      2.700000e+05
Name: annual_inc, dtype: float64


In [22]:
(df['annual_inc'] < 2000).sum()


np.int64(315)

In [23]:
import numpy as np

col = "annual_inc"

# 1️⃣ Identify the new sentinel rule (0 or <2000)
new_sentinel_mask = (df[col] == 0) | (df[col] < 2000)

# 2️⃣ Update the sentinel flag to include these
# If a row was already flagged, keep it as 1; otherwise add new True rows
df["annual_inc_was_sentinel"] = df["annual_inc_was_sentinel"] | new_sentinel_mask

# 3️⃣ Convert any new sentinel values (<2000) to NaN
df.loc[new_sentinel_mask, col] = np.nan

# 4️⃣ Recompute missing flag — mark all current NaN values
df["annual_inc_was_missing"] = df[col].isna().astype(int)

# 5️⃣ Quick check of what changed
print("Updated sentinel and missing counts:")
print("  Sentinel (0 or <2000):", df["annual_inc_was_sentinel"].sum())
print("  Missing (NaN) total:", df["annual_inc_was_missing"].sum())


Updated sentinel and missing counts:
  Sentinel (0 or <2000): 1982
  Missing (NaN) total: 2019


In [24]:
print("\nannual_inc percentiles after capping:")
print(df[col].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))


annual_inc percentiles after capping:
count    2.258682e+06
mean     7.622938e+04
std      4.457506e+04
min      2.000000e+03
1%       1.714317e+04
5%       2.800000e+04
50%      6.500000e+04
95%      1.600000e+05
99%      2.700000e+05
max      2.700000e+05
Name: annual_inc, dtype: float64


In [25]:
# --- Inspect DTI column for potential sentinels or extremes ---

col = "dti"

# Basic sanity check: how many nulls already present
missing_count = df[col].isna().sum()

# Suspicious or extreme conditions
above_43 = (df[col] > 43).sum()
equal_999 = (df[col] == 999).sum()
below_0 = (df[col] < 0).sum()
equal_neg1 = (df[col] == -1).sum()

# Display diagnostics
print(f"=== {col.upper()} Diagnostics ===")
print(f"Total rows: {len(df):,}")
print(f"Missing (NaN): {missing_count:,}")
print(f"Above 43: {above_43:,}")
print(f"Equal to 999: {equal_999:,}")
print(f"Below 0: {below_0:,}")
print(f"Equal to -1: {equal_neg1:,}")

# Quick check of extreme tail values for context
print("\nTop 10 highest DTI values:")
print(df[col].nlargest(20).values)
print("\nLowest 10 DTI values:")
print(df[col].nsmallest(20).values)


=== DTI Diagnostics ===
Total rows: 2,260,701
Missing (NaN): 1,744
Above 43: 22,150
Equal to 999: 135
Below 0: 2
Equal to -1: 2

Top 10 highest DTI values:
[999. 999. 999. 999. 999. 999. 999. 999. 999. 999. 999. 999. 999. 999.
 999. 999. 999. 999. 999. 999.]

Lowest 10 DTI values:
[-1. -1.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
  0.  0.]


In [32]:
col = "dti"
above_43 = (df[col] > 50).sum()
print(f"Above 43: {above_43:,}")

Above 43: 13,739


In [33]:
import numpy as np

col = "dti"

# 1. Sentinel (impossible values)
df["dti_was_sentinel"] = ((df[col] < 0) | (df[col] > 200)).astype(int)
df.loc[df["dti_was_sentinel"] == 1, col] = np.nan

# 2. Missing flag (after sentinel conversion)
df["dti_was_missing"] = df[col].isna().astype(int)

# 3. Extreme flag (values above 43 but not sentinel)
df["dti_was_extreme"] = ((df[col] > 43) & (~df[col].isna())).astype(int)

# 4. Cap extreme values at 43
df.loc[df["dti_was_extreme"] == 1, col] = 43

# 5. Quick check
print("DTI summary after cleaning:")
print(df[col].describe(percentiles=[0.01, 0.5, 0.99]))
print("\nFlag counts:")
for f in ["dti_was_sentinel", "dti_was_missing", "dti_was_extreme"]:
    print(f"{f}: {df[f].sum()}")


DTI summary after cleaning:
count    2.258119e+06
mean     1.847114e+01
std      8.831548e+00
min      0.000000e+00
1%       1.720000e+00
50%      1.783000e+01
99%      4.219000e+01
max      4.300000e+01
Name: dti, dtype: float64

Flag counts:
dti_was_sentinel: 838
dti_was_missing: 2582
dti_was_extreme: 21314


In [34]:
# === Revolving Utilization Diagnostics ===
col = "revol_util"

# Summary info
print(f"=== {col.upper()} Diagnostics ===")
print(f"Total rows: {len(df):,}")
print(f"Missing (NaN): {df[col].isna().sum():,}")

# Check key conditions
above_1_to_2 = df[(df[col] > 1) & (df[col] <= 2)].shape[0]
above_2 = df[df[col] > 2].shape[0]
max_val = df[col].max()
min_val = df[col].min()

print(f"Between 1 and 2: {above_1_to_2:,}")
print(f"Above 2: {above_2:,}")
print(f"Min: {min_val}")
print(f"Max: {max_val}")

# Optionally, look at top extreme values
print("\nTop 10 highest revol_util values:")
print(df[col].sort_values(ascending=False).head(10).values)

# Optional: show basic percentiles for reference
print("\nPercentiles (1%, 50%, 99%, 99.9%):")
print(df[col].quantile([0.01, 0.5, 0.99, 0.999]))


=== REVOL_UTIL Diagnostics ===
Total rows: 2,260,701
Missing (NaN): 1,835
Between 1 and 2: 7,341
Above 2: 2
Min: 0.0
Max: 8.923

Top 10 highest revol_util values:
[8.923 3.666 1.93  1.91  1.846 1.838 1.828 1.803 1.777 1.75 ]

Percentiles (1%, 50%, 99%, 99.9%):
0.010    0.009
0.500    0.503
0.990    0.981
0.999    1.021
Name: revol_util, dtype: float64


In [35]:
import numpy as np

col = "revol_util"

# 1️⃣ Sentinel handling (impossible values > 2)
df[f"{col}_was_sentinel"] = df[col] > 2
df.loc[df[f"{col}_was_sentinel"], col] = np.nan

# 2️⃣ Missing flag (includes sentinels)
df[f"{col}_was_missing"] = df[col].isna()

# 3️⃣ Extreme values (slightly above 1)
df[f"{col}_was_extreme"] = (df[col] > 1) & (df[col] <= 2)
df.loc[df[f"{col}_was_extreme"], col] = 1.0

# Diagnostics
total = len(df)
print(f"=== {col} diagnostics ===")
print(f"Total rows: {total:,}")
print(f"Sentinel (>2) converted to NaN: {df[f'{col}_was_sentinel'].sum():,}")
print(f"Missing (NaN) total: {df[f'{col}_was_missing'].sum():,}")
print(f"Extreme (1–2] capped to 1.0: {df[f'{col}_was_extreme'].sum():,}")
print(f"Min={df[col].min()}, Max={df[col].max()}")


=== revol_util diagnostics ===
Total rows: 2,260,701
Sentinel (>2) converted to NaN: 2
Missing (NaN) total: 1,837
Extreme (1–2] capped to 1.0: 7,341
Min=0.0, Max=1.0


In [37]:
col = "int_rate"

p1 = df[col].quantile(0.01)
p99 = df[col].quantile(0.99)
max_val = df[col].max()
min_val = df[col].min()

print(f"=== {col.upper()} Diagnostics ===")
print(f"1st percentile: {p1:.4f}")
print(f"99th percentile: {p99:.4f}")
print(f"Min: {min_val:.4f}")
print(f"Max: {max_val:.4f}")

# Check possible sentinels or outliers
above_99 = df[df[col] > p99]
below_1 = df[df[col] < p1]

print(f"\nValues above 99th percentile ({p99:.4f}): {len(above_99):,}")
print(f"Values below 1st percentile ({p1:.4f}): {len(below_1):,}")

# Check if any exact placeholder-looking values exist
print("\nTop unique high-end values:")
print(df[col].value_counts().head(20))

print("\nTop unique low-end values:")
print(df[col].value_counts(ascending=True).head(20))


=== INT_RATE Diagnostics ===
1st percentile: 0.0532
99th percentile: 0.2677
Min: 0.0531
Max: 0.3099

Values above 99th percentile (0.2677): 22,008
Values below 1st percentile (0.0532): 8,613

Top unique high-end values:
int_rate
0.1199    53869
0.0532    47171
0.1099    44165
0.1399    43025
0.1149    32010
0.1699    30564
0.1299    29276
0.0789    28514
0.0917    27835
0.1561    25208
0.1499    25108
0.1349    24304
0.1602    23711
0.1262    23421
0.1042    23182
0.0944    22982
0.1505    22283
0.0993    22268
0.1408    22145
0.1049    22020
Name: count, dtype: int64

Top unique low-end values:
int_rate
0.1428    1
0.1690    1
0.1872    1
0.1116    1
0.1746    1
0.1744    1
0.2440    1
0.2459    1
0.1778    1
0.1683    1
0.1741    1
0.2264    1
0.1750    1
0.1319    1
0.1772    1
0.2182    2
0.1384    2
0.1477    2
0.1633    2
0.2294    2
Name: count, dtype: int64


In [38]:
col = "int_rate"

# --- Check for exact system cap values (0.3099, or rounded versions like 0.31) ---
system_cap_vals = df[df[col].round(4) >= 0.3099]
print(f"\nRows with int_rate >= 0.3099: {len(system_cap_vals):,}")
print(system_cap_vals[col].value_counts().head(10))

# --- Check values above 0.28 (for tail distribution) ---
above_028 = df[df[col] > 0.28]
print(f"\nRows with int_rate > 0.28: {len(above_028):,}")

# --- Optional: see distribution of high-end interest rates ---
print("\nTop unique int_rate values above 0.28:")
print(df.loc[df[col] > 0.28, col].value_counts().sort_index(ascending=True).head(20))



Rows with int_rate >= 0.3099: 819
int_rate
0.3099    819
Name: count, dtype: int64

Rows with int_rate > 0.28: 17,036

Top unique int_rate values above 0.28:
int_rate
0.2814     224
0.2818     311
0.2834     195
0.2849     165
0.2867     140
0.2869    1415
0.2872    2248
0.2888     247
0.2899     284
0.2949     785
0.2967     178
0.2969    1214
0.2996     149
0.2999     641
0.3017    1236
0.3049     512
0.3065     983
0.3074     456
0.3075    1075
0.3079    1572
Name: count, dtype: int64


In [40]:
# === INT_RATE Final Cleaning ===

col = "int_rate"
p99 = 0.2677  # from  earlier analysis

# 1️⃣ Missing flag
df[f"{col}_was_missing"] = df[col].isna().astype("int8")

# 2️⃣ Extreme flag (values above 99th percentile)
df[f"{col}_was_extreme"] = (df[col] > p99).astype("int8")

# 3️⃣ Cap at 99th percentile
df.loc[df[col] > p99, col] = p99

# 4️⃣ Quick diagnostics
print(f">>> {col} diagnostics AFTER capping")
print(f"Missing (NaN): {df[f'{col}_was_missing'].sum():,}")
print(f"Extreme (>99p): {df[f'{col}_was_extreme'].sum():,}")
print(f"Max after capping: {df[col].max():.4f}")
print(f"99th percentile (should equal cap): {df[col].quantile(0.99):.4f}")


>>> int_rate diagnostics AFTER capping
Missing (NaN): 33
Extreme (>99p): 22,008
Max after capping: 0.2677
99th percentile (should equal cap): 0.2677


In [41]:
col = "installment"
p1 = df[col].quantile(0.01)
p99 = df[col].quantile(0.99)

print(f"=== {col.upper()} Diagnostics ===")
print(f"1st percentile: {p1:.2f}")
print(f"99th percentile: {p99:.2f}")
print(f"Min: {df[col].min():.2f}")
print(f"Max: {df[col].max():.2f}")
print(f"Missing (NaN): {df[col].isna().sum()}")

# Check extreme high values
extreme_high = df[df[col] > p99]
print(f"\nValues above 99th percentile ({p99:.2f}): {len(extreme_high):,}")
print("Top 10 highest occurring values above 99th percentile:")
print(extreme_high[col].value_counts().head(10))

# Check very low values
low_values = df[df[col] < p1]
print(f"\nValues below 1st percentile ({p1:.2f}): {len(low_values):,}")
print("Top 10 lowest occurring values below 1st percentile:")
print(low_values[col].value_counts().head(10))

# Check overall most frequent values (to detect repeated sentinels)
print("\nTop 10 most frequent installment values overall:")
print(df[col].value_counts().head(10))


=== INSTALLMENT Diagnostics ===
1st percentile: 53.46
99th percentile: 1238.46
Min: 4.93
Max: 1719.83
Missing (NaN): 33

Values above 99th percentile (1238.46): 22,582
Top 10 highest occurring values above 99th percentile:
installment
1247.68    898
1282.79    575
1238.93    545
1265.16    505
1280.20    464
1241.50    457
1252.91    440
1300.55    424
1269.73    416
1257.80    392
Name: count, dtype: int64

Values below 1st percentile (53.46): 22,605
Top 10 lowest occurring values below 1st percentile:
installment
33.21    288
34.18    237
32.98    205
33.94    192
38.92    191
34.69    189
35.17    178
33.57    177
51.26    167
33.52    164
Name: count, dtype: int64

Top 10 most frequent installment values overall:
installment
301.15    4420
332.10    4153
361.38    3704
327.34    3353
602.30    3095
451.73    3076
329.72    2614
166.05    2508
498.15    2410
180.69    2364
Name: count, dtype: int64


In [42]:
# === INSTALLMENT CLEANING ===

col = "installment"

# 1️⃣ Calculate key percentiles
p1, p99 = df[col].quantile([0.01, 0.99])

# 2️⃣ Create missing flag
df[f"{col}_was_missing"] = df[col].isna().astype("int8")

# 3️⃣ Create extreme flag (values <1st or >99th percentile)
df[f"{col}_was_extreme"] = ((df[col] < p1) | (df[col] > p99)).astype("int8")

# 4️⃣ Cap extremes (Winsorization)
df[col] = np.where(df[col] < p1, p1, df[col])
df[col] = np.where(df[col] > p99, p99, df[col])

# 5️⃣ Diagnostics summary
print(f"=== {col.upper()} Diagnostics ===")
print(f"1st percentile: {p1:.2f}")
print(f"99th percentile: {p99:.2f}")
print(f"Missing (NaN): {df[f'{col}_was_missing'].sum()}")
print(f"Extreme flagged: {df[f'{col}_was_extreme'].sum()}")
print(f"Min after capping: {df[col].min():.2f}")
print(f"Max after capping: {df[col].max():.2f}")
print(f"\nPercentiles after capping:\n{df[col].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])}")


=== INSTALLMENT Diagnostics ===
1st percentile: 53.46
99th percentile: 1238.46
Missing (NaN): 33
Extreme flagged: 45187
Min after capping: 53.46
Max after capping: 1238.46

Percentiles after capping:
count    2.260668e+06
mean     4.453342e+02
std      2.650785e+02
min      5.346000e+01
1%       5.346000e+01
5%       1.104300e+02
50%      3.779900e+02
95%      9.844700e+02
99%      1.238460e+03
max      1.238460e+03
Name: installment, dtype: float64


In [43]:
cols = ["loan_amnt", "funded_amnt"]

for col in cols:
    print(f"\n=== {col.upper()} Diagnostics ===")
    p1 = df[col].quantile(0.01)
    p99 = df[col].quantile(0.99)
    
    print(f"1st percentile: {p1:.2f}")
    print(f"99th percentile: {p99:.2f}")
    print(f"Min: {df[col].min():.2f}")
    print(f"Max: {df[col].max():.2f}")
    print(f"Missing (NaN): {df[col].isna().sum():,}")

    below_1p = df[df[col] < p1]
    print(f"Values below 1st percentile ({p1:.2f}): {len(below_1p):,}")
    if len(below_1p) > 0:
        print("\nTop 10 most frequent values below 1st percentile:")
        print(below_1p[col].value_counts().head(10))

    print("\nLowest 5 unique values overall:")
    print(sorted(df[col].unique())[:5])



=== LOAN_AMNT Diagnostics ===
1st percentile: 1525.00
99th percentile: 40000.00
Min: 500.00
Max: 40000.00
Missing (NaN): 33
Values below 1st percentile (1525.00): 22,595

Top 10 most frequent values below 1st percentile:
loan_amnt
1000.0    9816
1500.0    6176
1200.0    3691
1400.0    1265
1300.0     375
1450.0     346
1100.0     252
1250.0     101
1325.0      74
1350.0      67
Name: count, dtype: int64

Lowest 5 unique values overall:
[np.float64(500.0), np.float64(1150.0), np.float64(1350.0), np.float64(1425.0), np.float64(1525.0)]

=== FUNDED_AMNT Diagnostics ===
1st percentile: 1516.75
99th percentile: 40000.00
Min: 500.00
Max: 40000.00
Missing (NaN): 33
Values below 1st percentile (1516.75): 22,607

Top 10 most frequent values below 1st percentile:
funded_amnt
1000.0    9817
1500.0    6175
1200.0    3696
1400.0    1264
1300.0     375
1450.0     346
1100.0     253
1250.0     102
1325.0      74
1350.0      67
Name: count, dtype: int64

Lowest 5 unique values overall:
[np.float64(50

In [44]:
import numpy as np

cols = ["loan_amnt", "funded_amnt"]

for col in cols:
    print(f"\n=== Processing {col.upper()} ===")

    p1 = df[col].quantile(0.01)
    p99 = df[col].quantile(0.99)

    # 1️⃣ Missing flag
    df[f"{col}_was_missing"] = df[col].isna().astype("int8")

    # 2️⃣ Extreme flag — values below 1st percentile
    df[f"{col}_was_extreme"] = (df[col] < p1).astype("int8")

    # 3️⃣ Cap low values (below 1st percentile)
    df.loc[df[col] < p1, col] = p1

    # 4️⃣ Confirm results
    print(f"Capped {df[f'{col}_was_extreme'].sum():,} values below {p1:.2f}")
    print(f"Min after capping: {df[col].min():.2f}")
    print(f"Missing count: {df[f'{col}_was_missing'].sum():,}")

#  check
df[["loan_amnt", "loan_amnt_was_missing", "loan_amnt_was_extreme",
    "funded_amnt", "funded_amnt_was_missing", "funded_amnt_was_extreme"]].head()



=== Processing LOAN_AMNT ===
Capped 22,595 values below 1525.00
Min after capping: 1525.00
Missing count: 33

=== Processing FUNDED_AMNT ===
Capped 22,607 values below 1516.75
Min after capping: 1516.75
Missing count: 33


,loan_amnt,loan_amnt_was_missing,loan_amnt_was_extreme,funded_amnt,funded_amnt_was_missing,funded_amnt_was_extreme
0,3600.0,0,0,3600.0,0,0
1,24700.0,0,0,24700.0,0,0
2,20000.0,0,0,20000.0,0,0
3,35000.0,0,0,35000.0,0,0
4,10400.0,0,0,10400.0,0,0


In [45]:
# : Create fico_mean (average of low/high)
df["fico_mean"] = ((df["fico_range_low"] + df["fico_range_high"]) / 2).round().astype("Int64")

print("✅ Created 'fico_mean' column.")
print(df["fico_mean"].head(10))
print(df["fico_mean"].dtype)


✅ Created 'fico_mean' column.
0    677
1    717
2    697
3    787
4    697
5    692
6    682
7    707
8    687
9    702
Name: fico_mean, dtype: Int64
Int64


In [46]:
#  Inspect FICO Mean distribution
import numpy as np

col = "fico_mean"

# Basic stats with custom percentiles
desc = df[col].describe(percentiles=[0.01, 0.5, 0.99])
print("\n=== FICO_MEAN Summary Statistics ===")
print(desc)

# Count missing values
missing_count = df[col].isna().sum()

# Count extreme values above 99p and below 1p
p1, p99 = desc["1%"], desc["99%"]
above_99p = (df[col] > p99).sum()
below_1p = (df[col] < p1).sum()

print(f"\nMissing (NaN): {missing_count:,}")
print(f"Values below 1st percentile ({p1:.0f}): {below_1p:,}")
print(f"Values above 99th percentile ({p99:.0f}): {above_99p:,}")

#  check min/max for sanity
print(f"\nMin: {df[col].min()}, Max: {df[col].max()}, Std: {df[col].std():.2f}")



=== FICO_MEAN Summary Statistics ===
count    2260668.0
mean      700.5884
std      33.011245
min          612.0
1%           662.0
50%          692.0
99%          807.0
max          848.0
Name: fico_mean, dtype: Float64

Missing (NaN): 33
Values below 1st percentile (662): 489
Values above 99th percentile (807): 16,890

Min: 612, Max: 848, Std: 33.01


In [47]:
# === Handle Missing FICO_MEAN ===

# 1️⃣ Create missing flag
df["fico_mean_was_missing"] = df["fico_mean"].isna().astype("int8")

# 2️⃣ Optional — check how many rows are missing
missing_count = df["fico_mean_was_missing"].sum()
total_rows = len(df)
print(f"Missing (NaN) FICO_MEAN rows flagged: {missing_count:,} ({missing_count/total_rows:.3%})")

# 3️⃣ Preview a few rows
print(df[["fico_mean", "fico_mean_was_missing"]].head(10))


Missing (NaN) FICO_MEAN rows flagged: 33 (0.001%)
   fico_mean  fico_mean_was_missing
0        677                      0
1        717                      0
2        697                      0
3        787                      0
4        697                      0
5        692                      0
6        682                      0
7        707                      0
8        687                      0
9        702                      0


In [48]:
# === EMP_LENGTH_YRS Diagnostics ===
col = "emp_length_yrs"

print(f"=== {col.upper()} Diagnostics ===")

# 1. Missing values
missing_count = df[col].isna().sum()
total_rows = len(df)
print(f"Total rows: {total_rows:,}")
print(f"Missing (NaN): {missing_count:,} ({100*missing_count/total_rows:.2f}%)")

# 2. Top 10 most frequent values
print("\nTop 10 most frequent values:")
print(df[col].value_counts(dropna=False).head(10))


=== EMP_LENGTH_YRS Diagnostics ===
Total rows: 2,260,701
Missing (NaN): 146,940 (6.50%)

Top 10 most frequent values:
emp_length_yrs
10      748005
2       203677
0       189988
3       180753
1       148403
<NA>    146940
5       139698
4       136605
6       102628
7        92695
Name: count, dtype: Int64


In [49]:
col = "emp_length_yrs"

# Create missing flag (int8 )
df["emp_length_was_missing"] = df[col].isna().astype("int8")

print(f"{col}: Missing flagged {df['emp_length_was_missing'].sum():,} rows ({100*df['emp_length_was_missing'].mean():.2f}%)")


emp_length_yrs: Missing flagged 146,940 rows (6.50%)


In [50]:
# === TERM_MONTHS Diagnostics ===
col = 'term_months'

print(f"\n=== {col.upper()} Diagnostics ===")

# 1. Unique values and their counts (including NaN)
print("\nUnique values and their frequencies (including NaN):")
print(df[col].value_counts(dropna=False))

# 2. Count missing
missing_count = df[col].isna().sum()
total = len(df)
print(f"\nMissing (NaN): {missing_count} ({100 * missing_count / total:.2f}%)")

# 3. Summary stats
print("\nSummary statistics:")
print(df[col].describe())



=== TERM_MONTHS Diagnostics ===

Unique values and their frequencies (including NaN):
term_months
36      1609754
60       650914
<NA>         33
Name: count, dtype: Int64

Missing (NaN): 33 (0.00%)

Summary statistics:
count    2260668.0
mean     42.910319
std      10.867161
min           36.0
25%           36.0
50%           36.0
75%           60.0
max           60.0
Name: term_months, dtype: Float64


In [51]:
# === TERM_MONTHS Cleaning ===
col = "term_months"

# Create missing flag
df[f"{col}_was_missing"] = df[col].isna().astype("int8")

# Quick check
print(f"{col}_was_missing created — Missing count: {df[f'{col}_was_missing'].sum()} ({100 * df[f'{col}_was_missing'].mean():.4f}%)")


term_months_was_missing created — Missing count: 33 (0.0015%)


In [56]:
# === CREDIT_HISTORY_YEARS Detailed Inspection ===
col = "credit_history_years"

# Calculate percentiles
p1 = df[col].quantile(0.01)
p99 = df[col].quantile(0.99)

# Basic diagnostics
missing_count = df[col].isna().sum()
below_1p = (df[col] < p1).sum()
above_99p = (df[col] > p99).sum()
above_50 = (df[col] > 50).sum()
above_60 = (df[col] > 60).sum()
above_70 = (df[col] > 70).sum()
above_80 = (df[col] > 80).sum()

print(f"=== {col.upper()} Diagnostics ===")
print(f"1st percentile: {p1:.2f}")
print(f"99th percentile: {p99:.2f}\n")

print(f"Missing (NaN): {missing_count}")
print(f"Values below 1st percentile ({p1:.2f}): {below_1p}")
print(f"Values above 99th percentile ({p99:.2f}): {above_99p}")
print(f"Values above 50: {above_50}")
print(f"Values above 60: {above_60}")
print(f"Values above 70: {above_70}")
print(f"Values above 80: {above_80}")

# Top 20 most frequent values overall
print("\nTop 20 most frequent values:")
print(df[col].value_counts(dropna=False).head(20))



=== CREDIT_HISTORY_YEARS Diagnostics ===
1st percentile: 3.75
99th percentile: 40.41

Missing (NaN): 62
Values below 1st percentile (3.75): 21713
Values above 99th percentile (40.41): 22589
Values above 50: 2482
Values above 60: 117
Values above 70: 9
Values above 80: 4

Top 20 most frequent values:
credit_history_years
12.000000    13802
12.999316    11020
16.000000    10209
11.000684    10126
12.167009     9989
11.832991     9349
15.000684     9254
11.748118     8624
11.915127     8473
12.418891     8328
12.251882     8219
11.581109     8065
12.832307     7980
13.166324     7874
11.167693     7740
12.084873     7677
15.832991     7083
16.999316     7081
16.167009     7056
12.747433     7016
Name: count, dtype: int64


In [57]:
# === CREDIT_HISTORY_YEARS Processing ===

col = "credit_history_years"
p1, p99 = 3.75, 40.41  # Based on your earlier stats

# 1️⃣ Sentinel detection (values above 80 years — impossible)
df["credit_history_was_sentinel"] = (df[col] > 80).astype("int8")
sentinel_count = df["credit_history_was_sentinel"].sum()
print(f"Sentinel (>80 yrs): {sentinel_count:,} rows")

# Convert sentinel to NaN
df.loc[df["credit_history_was_sentinel"] == 1, col] = np.nan

# 2️⃣ Missing flag (includes sentinel now)
df["credit_history_was_missing"] = df[col].isna().astype("int8")
missing_count = df["credit_history_was_missing"].sum()
print(f"Missing (NaN, incl. sentinel): {missing_count:,} rows")

# 3️⃣ Extreme flag (both sides)
df["credit_history_was_extreme"] = (
    (df[col] < p1) | (df[col] > p99)
).astype("int8")
extreme_count = df["credit_history_was_extreme"].sum()
print(f"Extreme (<{p1} or >{p99}): {extreme_count:,} rows")

# 4️⃣  flag very long credit history (super-prime customers)
df["credit_history_was_very_long"] = (df[col] > p99).astype("int8")
very_long_count = df["credit_history_was_very_long"].sum()
print(f"Very long credit history (> {p99} yrs): {very_long_count:,} rows")

# 5️⃣ Cap only below 1st percentile
below_p1 = (df[col] < p1).sum()
df.loc[df[col] < p1, col] = p1
print(f"Capped {below_p1:,} values below {p1}")

# 6️⃣ Post-capping diagnostics
print("\n=== CREDIT_HISTORY_YEARS After Capping ===")
print(df[col].describe(percentiles=[0.01, 0.5, 0.99]))
print("\nFlag counts:")
print(df[[
    "credit_history_was_sentinel",
    "credit_history_was_missing",
    "credit_history_was_extreme",
    "credit_history_was_very_long"
]].sum())


Sentinel (>80 yrs): 4 rows
Missing (NaN, incl. sentinel): 66 rows
Extreme (<3.75 or >40.41): 46,290 rows
Very long credit history (> 40.41 yrs): 22,708 rows
Capped 23,582 values below 3.75

=== CREDIT_HISTORY_YEARS After Capping ===
count    2.260635e+06
mean     1.639804e+01
std      7.673242e+00
min      3.750000e+00
1%       3.750000e+00
50%      1.483368e+01
99%      4.041342e+01
max      7.508556e+01
Name: credit_history_years, dtype: float64

Flag counts:
credit_history_was_sentinel         4
credit_history_was_missing         66
credit_history_was_extreme      46290
credit_history_was_very_long    22708
dtype: int64


In [61]:
col = "open_acc"

print(f"=== {col.upper()} Diagnostics ===")

# Basic percentiles
p1 = df[col].quantile(0.01)
p99 = df[col].quantile(0.99)
print(f"1st percentile: {p1:.2f}")
print(f"99th percentile: {p99:.2f}")
print(f"Min: {df[col].min()}, Max: {df[col].max()}")

# Missing values
missing = df[col].isna().sum()
print(f"\nMissing (NaN): {missing} ({missing/len(df):.2%})")

# Below/above percentile thresholds
below_1p = df.loc[df[col] < p1, col].count()
above_99p = df.loc[df[col] > p99, col].count()
print(f"Values below 1st percentile ({p1:.2f}): {below_1p}")
print(f"Values above 99th percentile ({p99:.2f}): {above_99p}")



# Optional: inspect suspiciously high values (e.g., >60)
above_60 = df.loc[df[col] == 101, col].count()
print(f"Values >101 (potential extreme tail): {above_60}")

# Top 20 most frequent values overall
print("\nTop 20 most frequent values:")
print(df[col].value_counts(dropna=False).head(20))


=== OPEN_ACC Diagnostics ===
1st percentile: 3.00
99th percentile: 30.00
Min: 0, Max: 101

Missing (NaN): 62 (0.00%)
Values below 1st percentile (3.00): 12560
Values above 99th percentile (30.00): 18921
Values >101 (potential extreme tail): 1

Top 20 most frequent values:
open_acc
9     195762
10    189737
8     188717
11    175101
7     172834
12    157331
6     145444
13    137502
14    118314
5     108565
15     99759
16     83739
17     69542
4      67827
18     57601
19     47510
20     38373
3      32428
21     31184
22     25422
Name: count, dtype: Int64


In [62]:
# === OPEN_ACC Diagnostics and Flagging ===

col = "open_acc"
p1, p99 = df[col].quantile([0.01, 0.99])
max_val = df[col].max()

print(f"\n=== {col.upper()} Diagnostics ===")
print(f"1st percentile: {p1:.2f}")
print(f"99th percentile: {p99:.2f}")
print(f"Min: {df[col].min()}")
print(f"Max: {max_val}")
print(f"Missing (NaN): {df[col].isna().sum()}")

# --- Create Flags ---
df[f"{col}_was_missing"] = df[col].isna().astype("int8")

# Extreme = anything below 1st or above 99th percentile
df[f"{col}_was_extreme"] = (
    (df[col] < p1) | (df[col] > p99)
).astype("int8")

print("\nFlag counts:")
print(df[[f"{col}_was_missing", f"{col}_was_extreme"]].sum())

# --- Summary of tail extremes ---
print("\nTop 10 highest open_acc values:")
print(df.loc[df[col] > p99, col].value_counts().head(10))

print("\nBottom 10 lowest open_acc values:")
print(df.loc[df[col] < p1, col].value_counts().head(10))

print("\nPercentiles after flagging (no capping applied):")
print(df[col].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))



=== OPEN_ACC Diagnostics ===
1st percentile: 3.00
99th percentile: 30.00
Min: 0
Max: 101
Missing (NaN): 62


ValueError: cannot convert NA to integer

In [64]:
for col in ["open_acc_was_missing", "open_acc_was_extreme"]:
    if col in df.columns:
        df.drop(columns=col, inplace=True)
print('done')

done


In [65]:
col = "open_acc"

# 1st & 99th percentile
p1, p99 = df[col].quantile([0.01, 0.99])

# Flag missing values
df[f"{col}_was_missing"] = df[col].isna().astype("int8")

# Flag extremes safely
df[f"{col}_was_extreme"] = (
    ((df[col] < p1) | (df[col] > p99))
    .fillna(False)     # Missing → False
    .astype("int8")    # Convert True/False → 1/0
)

print(f"\nFlags created for '{col}':")
print(df[[f"{col}_was_missing", f"{col}_was_extreme"]].sum())
print(f"1st percentile: {p1:.2f}, 99th percentile: {p99:.2f}")



Flags created for 'open_acc':
open_acc_was_missing       62
open_acc_was_extreme    31481
dtype: int64
1st percentile: 3.00, 99th percentile: 30.00


In [72]:
col = "pub_rec"
p1, p99 = df[col].quantile([0.01, 0.99])
below_1p = df[df[col] < p1]
above_99p = df[df[col] > p99]
print(f"\nValues below 1st percentile ({p1:.2f}): {len(below_1p):,}")
print(f"Values above 99th percentile ({p99:.2f}): {len(above_99p):,}")
print(f"Missing (NaN): {df[col].isna().sum()}")
print("\nTop 10 most frequent values overall:")
print(df[col].value_counts(dropna=False).head(10))

high_40 = df[df[col] > 15]
print(f"Values > 10: {len(high_40):,}")
if len(high_40) > 0:
    print("\nTop 10 extreme values above 40:")
    print(high_40[col].value_counts().head(10))


Values below 1st percentile (0.00): 0
Values above 99th percentile (2.00): 18,337
Missing (NaN): 62

Top 10 most frequent values overall:
pub_rec
0    1902758
1     305390
2      34154
3      10567
4       3872
5       1843
6        933
7        427
8        243
9        143
Name: count, dtype: Int64
Values > 10: 72

Top 10 extreme values above 40:
pub_rec
16    11
19     9
18     6
21     6
17     5
28     4
24     4
22     3
20     3
61     2
Name: count, dtype: Int64


In [76]:
col = "pub_rec"

# Drop old flags if re-running
for flag in [f"{col}_was_sentinel", f"{col}_was_missing", f"{col}_was_extreme"]:
    if flag in df.columns:
        df.drop(columns=flag, inplace=True)

# Sentinel: pub_rec > 40
df[f"{col}_was_sentinel"] = (df[col] > 40).fillna(False).astype("int8")

# Missing flag (includes sentinel values too, so they are still identifiable)
df[f"{col}_was_missing"] = df[col].isna().astype("int8")

# Extreme flag: values above 99th percentile (2)
p99 = df[col].quantile(0.99)
df[f"{col}_was_extreme"] = (df[col] > p99).fillna(False).astype("int8")

print(f"=== {col.upper()} Flag Summary ===")
print(f"99th percentile: {p99:.2f}")
print(df[[f"{col}_was_sentinel", f"{col}_was_missing", f"{col}_was_extreme"]].sum())

true_nan = df[col].isna().sum()
sentinel_count = df[f"{col}_was_sentinel"].sum()
extreme_count = df[f"{col}_was_extreme"].sum()
flagged_missing = df[f"{col}_was_missing"].sum()



=== PUB_REC Flag Summary ===
99th percentile: 2.00
pub_rec_was_sentinel       12
pub_rec_was_missing        62
pub_rec_was_extreme     18337
dtype: int64


In [77]:
# === Clean reset & recreate flags for pub_rec ===
col = "pub_rec"

# Drop old flags if they exist
for flag in [f"{col}_was_sentinel", f"{col}_was_missing", f"{col}_was_extreme"]:
    if flag in df.columns:
        df.drop(columns=flag, inplace=True)

# Compute the 99th percentile for reference
p99 = df[col].quantile(0.99)

# 1️⃣ Sentinel flag: pub_rec > 40 (DO NOT convert to NaN)
df[f"{col}_was_sentinel"] = (df[col] > 40).fillna(False).astype("int8")

# 2️⃣ Missing flag: only true NaNs (exclude sentinels)
df[f"{col}_was_missing"] = df[col].isna().astype("int8")

# 3️⃣ Extreme flag: values above 99th percentile (NaNs treated as False)
df[f"{col}_was_extreme"] = (df[col] > p99).fillna(False).astype("int8")

# --- ✅ Verification ---
true_nan = df[col].isna().sum()
sentinel_count = df[f"{col}_was_sentinel"].sum()
extreme_count = df[f"{col}_was_extreme"].sum()
flagged_missing = df[f"{col}_was_missing"].sum()
sentinel_are_nan = df.loc[df[f"{col}_was_sentinel"] == 1, col].isna().sum()

print("=== PUB_REC Flags Verification ===")
print(f"True NaNs: {true_nan}")
print(f"Sentinel (>40): {sentinel_count}")
print(f"Extreme (>99p={p99:.2f}): {extreme_count}")
print(f"Flagged Missing (should equal true NaNs): {flagged_missing}")
print(f"Sentinel rows that are NaN (should be 0): {sentinel_are_nan}")



=== PUB_REC Flags Verification ===
True NaNs: 62
Sentinel (>40): 12
Extreme (>99p=2.00): 18337
Flagged Missing (should equal true NaNs): 62
Sentinel rows that are NaN (should be 0): 0


In [78]:
col = "inq_last_6mths"

p1 = df[col].quantile(0.01)
p99 = df[col].quantile(0.99)

print(f"=== {col.upper()} Diagnostics ===")
print(f"1st percentile: {p1:.2f}")
print(f"99th percentile: {p99:.2f}")

print(f"\nMissing (NaN): {df[col].isna().sum()}")
print(f"Values below 1st percentile ({p1:.2f}): {(df[col] < p1).sum()}")
print(f"Values above 99th percentile ({p99:.2f}): {(df[col] > p99).sum()}")
print(f"Values above 10: {(df[col] > 10).sum()}")
print(f"Values above 20: {(df[col] > 20).sum()}")

print("\nTop 20 most frequent values:")
print(df[col].value_counts(dropna=False).head(20))


=== INQ_LAST_6MTHS Diagnostics ===
1st percentile: 0.00
99th percentile: 4.00

Missing (NaN): 63
Values below 1st percentile (0.00): 0
Values above 99th percentile (4.00): 7925
Values above 10: 71
Values above 20: 8

Top 20 most frequent values:
inq_last_6mths
0       1381722
1        584390
2        200212
3         69009
4         17380
5          6232
6          1231
7           195
8           122
<NA>         63
9            50
10           24
11           15
12           15
15            9
13            6
14            6
18            4
16            3
17            2
Name: count, dtype: Int64


In [79]:
col = "inq_last_6mths"

p1 = df[col].quantile(0.01)
p99 = df[col].quantile(0.99)

# 1️⃣ Sentinel flag — unrealistic high values
df[f"{col}_was_sentinel"] = (df[col] > 20).fillna(False).astype("int8")

# 2️⃣ Missing flag — actual NaNs only
df[f"{col}_was_missing"] = df[col].isna().astype("int8")

# 3️⃣ Extreme flag — below 1p or above 99p (NaN treated as False)
df[f"{col}_was_extreme"] = ((df[col] < p1) | (df[col] > p99)).fillna(False).astype("int8")

#  Verification summary
print(f"=== {col.upper()} Flag Summary ===")
print(f"1st percentile: {p1:.2f}, 99th percentile: {p99:.2f}")
print(df[[f"{col}_was_sentinel", f"{col}_was_missing", f"{col}_was_extreme"]].sum())

print("\nSentinel values (>20):")
print(df.loc[df[col] > 20, col].value_counts())


=== INQ_LAST_6MTHS Flag Summary ===
1st percentile: 0.00, 99th percentile: 4.00
inq_last_6mths_was_sentinel       8
inq_last_6mths_was_missing       63
inq_last_6mths_was_extreme     7925
dtype: int64

Sentinel values (>20):
inq_last_6mths
24    2
33    1
32    1
31    1
28    1
25    1
27    1
Name: count, dtype: Int64


In [81]:
col = "total_acc"

# Compute percentiles for reference
p1, p99 = df[col].quantile([0.01, 0.99])
print(f"1st percentile: {p1:.2f}")
print(f"99th percentile: {p99:.2f}")

# --- Diagnostics ---
print("\n=== TOTAL_ACC Diagnostics ===")

# Missing values
missing_count = df[col].isna().sum()
print(f"Missing (NaN): {missing_count:,}")

# Values below 1st percentile
below_1p = (df[col] < p1).sum()
print(f"Values below 1st percentile ({p1:.2f}): {below_1p:,}")

# Values above 99th percentile
above_99p = (df[col] > p99).sum()
print(f"Values above 99th percentile ({p99:.2f}): {above_99p:,}")

# Additional checks for possible sentinel/extreme zones
above_100 = (df[col] > 100).sum()
above_150 = (df[col] > 150).sum()
print(f"Values above 100: {above_100:,}")
print(f"Values above 150: {above_150:,}")

# Top 20 most frequent values overall
print("\nTop 20 most frequent values:")
print(df[col].value_counts(dropna=False).head(20))


1st percentile: 5.00
99th percentile: 60.00

=== TOTAL_ACC Diagnostics ===
Missing (NaN): 62
Values below 1st percentile (5.00): 16,054
Values above 99th percentile (60.00): 22,079
Values above 100: 6,340
Values above 150: 2,222

Top 20 most frequent values:
total_acc
20    82570
19    82012
18    81931
17    81378
21    81170
16    79655
22    79438
23    77691
15    77146
24    75330
14    74211
25    72476
13    70611
26    68903
27    65726
12    65217
28    61547
11    59705
29    58460
30    55123
Name: count, dtype: Int64


In [82]:
col = "total_acc"

# Get percentile cutoffs
p1, p99 = df[col].quantile([0.01, 0.99])

# --- Step 1: Sentinel flag ---
df[f"{col}_was_sentinel"] = (df[col] > 150).fillna(False).astype("int8")

# --- Step 2: Missing flag ---
df[f"{col}_was_missing"] = df[col].isna().astype("int8")

# --- Step 3: Extreme flag (<1p or >99p) ---
df[f"{col}_was_extreme"] = ((df[col] < p1) | (df[col] > p99)).fillna(False).astype("int8")

# --- Step 4: Quick verification ---
print(f"=== {col.upper()} Flag Summary ===")
print(f"1st percentile: {p1:.2f}, 99th percentile: {p99:.2f}")
print(df[[f"{col}_was_sentinel", f"{col}_was_missing", f"{col}_was_extreme"]].sum())

# Check sentinel value distribution
print("\nSentinel values (>150):")
print(df.loc[df[col] > 150, col].value_counts())


=== TOTAL_ACC Flag Summary ===
1st percentile: 5.00, 99th percentile: 60.00
total_acc_was_sentinel       13
total_acc_was_missing        62
total_acc_was_extreme     38133
dtype: int64

Sentinel values (>150):
total_acc
151    3
160    2
169    1
162    1
153    1
173    1
157    1
176    1
156    1
165    1
Name: count, dtype: Int64


In [88]:
col = "delinq_2yrs"

print(f"=== {col.upper()} Diagnostics ===")

# --- Percentiles ---
p1, p99 = df[col].quantile([0.01, 0.99])
print(f"1st percentile: {p1:.2f}")
print(f"99th percentile: {p99:.2f}")

# --- Missing values ---
missing_count = df[col].isna().sum()
print(f"\nMissing (NaN): {missing_count}")

# --- Extremes ---
above_99p = (df[col] > p99).sum()
below_1p = (df[col] < p1).sum()
above_20 = (df[col] > 20).sum()
above_50 = (df[col] > 25).sum()

print(f"Values above 99th percentile ({p99:.2f}): {above_99p}")
print(f"Values abelow 1st percentile ({p1:.2f}): {below_1p}")
print(f"Values above 20: {above_20}")
print(f"Values above 25: {above_50}")

# --- Top 20 most frequent values ---
print("\nTop 20 most frequent values:")
print(df[col].value_counts(dropna=False).head(20))


=== DELINQ_2YRS Diagnostics ===
1st percentile: 0.00
99th percentile: 4.00

Missing (NaN): 62
Values above 99th percentile (4.00): 16168
Values abelow 1st percentile (0.00): 0
Values above 20: 40
Values above 25: 15

Top 20 most frequent values:
delinq_2yrs
0       1839108
1        281353
2         81289
3         29542
4         13179
5          6599
6          3717
7          2062
8          1223
9           818
10          556
11          363
12          263
13          165
14          120
15           87
<NA>         62
16           55
18           30
17           30
Name: count, dtype: Int64


In [89]:
# === DELINQ_2YRS Flags ===
col = "delinq_2yrs"

# Get percentile cutoffs
p1, p99 = df[col].quantile([0.01, 0.99])

# --- Step 1: Sentinel flag (>50) ---
df[f"{col}_was_sentinel"] = (df[col] > 50).fillna(False).astype("int8")

# --- Step 2: Missing flag ---
df[f"{col}_was_missing"] = df[col].isna().astype("int8")

# --- Step 3: Extreme flag (above 99th percentile) ---
df[f"{col}_was_extreme"] = (df[col] > p99).fillna(False).astype("int8")

# --- Step 4: Verification summary ---
print(f"=== {col.upper()} Flag Summary ===")
print(f"1st percentile: {p1:.2f}, 99th percentile: {p99:.2f}")
print(df[[f"{col}_was_sentinel", f"{col}_was_missing", f"{col}_was_extreme"]].sum())

# Sentinel value breakdown
print("\nSentinel values (>50):")
print(df.loc[df[col] > 50, col].value_counts())

# Quick validation
true_nans = df[col].isna().sum()
flagged_missing = df[f"{col}_was_missing"].sum()
print(f"\nTrue NaNs: {true_nans}, Flagged Missing: {flagged_missing} (should match)")


=== DELINQ_2YRS Flag Summary ===
1st percentile: 0.00, 99th percentile: 4.00
delinq_2yrs_was_sentinel        1
delinq_2yrs_was_missing        62
delinq_2yrs_was_extreme     16168
dtype: int64

Sentinel values (>50):
delinq_2yrs
58    1
Name: count, dtype: Int64

True NaNs: 62, Flagged Missing: 62 (should match)


In [90]:
# List of your 8 categorical columns
cat_cols = [
    "grade", "sub_grade", "home_ownership", "loan_status",
    "purpose", "addr_state", "application_type", "default_flag"
]

# Check dtype for each column
print("=== DTYPE CHECK FOR CATEGORICAL COLUMNS ===")
for col in cat_cols:
    dtype = df[col].dtype
    print(f"{col:<20} -> {dtype}")


=== DTYPE CHECK FOR CATEGORICAL COLUMNS ===
grade                -> category
sub_grade            -> category
home_ownership       -> category
loan_status          -> category
purpose              -> category
addr_state           -> category
application_type     -> category
default_flag         -> Int64


In [91]:
# -------------------------
# Categorical cleaning (batch)
# -------------------------
import pandas as pd

cat_cols = [
    "grade", "sub_grade", "home_ownership", "loan_status",
    "purpose", "addr_state", "application_type"
]

# threshold for considering a label 'rare' (proportion). Default 0.005 = 0.5%
rare_thresh = 0.005

# Strings we treat as missing/unknown (case-insensitive)
missing_variants = {"", "none", "n/a", "na", "unknown", "other", "nan", "null", "missing"}

print("Running categorical cleanup for columns:", ", ".join(cat_cols))
print("Rare-label threshold (fraction):", rare_thresh)
print("Missing-like variants (case-insensitive):", sorted(missing_variants))
print("")

for col in cat_cols:
    if col not in df.columns:
        print(f"⚠️ Column '{col}' not found in DataFrame — skipping.")
        continue

    print(f"--- Processing column: {col} ---")

    # 1) Work on a string view for detection (safe for both category and object dtypes)
    s_str = df[col].astype(str).str.strip()             # e.g., NaN -> 'nan'
    s_lower = s_str.str.lower()

    # 2) Mask of values we consider missing/unknown BEFORE any replacement
    mask_missing_like = s_lower.isin(missing_variants)

    # 3) Create missing flag from that mask (1 if missing-like, else 0)
    missing_flag_col = f"{col}_was_missing"
    df[missing_flag_col] = mask_missing_like.astype("int8")

    # 4) Add 'Unknown' and 'Other' to categories if column is categorical
    if pd.api.types.is_categorical_dtype(df[col]):
        # ensure categories exist first
        for newcat in ("Unknown", "Other"):
            if newcat not in df[col].cat.categories:
                df[col] = df[col].cat.add_categories([newcat])
    else:
        # if not categorical, we'll convert to category at the end
        pass

    # 5) Replace detected missing-like labels with 'Unknown'
    #    (This converts string forms like 'nan','none','unknown' to 'Unknown')
    df.loc[mask_missing_like, col] = "Unknown"

    # 6) Also ensure any true NaN (if any left) become 'Unknown'
    df[col] = df[col].fillna("Unknown")

    # 7) Now combine rare labels into 'Other'
    #    - compute frequencies AFTER mapping missing -> 'Unknown'
    freqs = df[col].value_counts(normalize=True, dropna=False)
    # identify labels below threshold but do NOT include 'Unknown' or 'Other'
    rare_labels = [lab for lab, frac in freqs.items()
                   if (frac < rare_thresh) and (lab not in ("Unknown", "Other"))]

    if rare_labels:
        # ensure 'Other' is in categories
        if pd.api.types.is_categorical_dtype(df[col]):
            if "Other" not in df[col].cat.categories:
                df[col] = df[col].cat.add_categories(["Other"])
        # perform replacement
        df[col] = df[col].replace(rare_labels, "Other")

    # 8) Ensure dtype is category (if not, convert)
    if not pd.api.types.is_categorical_dtype(df[col]):
        df[col] = df[col].astype("category")

    # 9) Report summary for human verification
    n_unique = df[col].nunique(dropna=True)
    missing_count = int(df[missing_flag_col].sum())
    unknown_count = int((df[col] == "Unknown").sum())
    other_count = int((df[col] == "Other").sum())
    top_vals = df[col].value_counts().head(8)

    print(f"Unique categories after cleaning: {n_unique}")
    print(f"{missing_flag_col} (count): {missing_count}")
    print(f"Count 'Unknown': {unknown_count}, Count 'Other': {other_count}")
    print("Top categories (after grouping):")
    print(top_vals)
    print("")

print("Categorical cleanup complete. All processed columns are category dtype and have missing flags.")


Running categorical cleanup for columns: grade, sub_grade, home_ownership, loan_status, purpose, addr_state, application_type
Rare-label threshold (fraction): 0.005
Missing-like variants (case-insensitive): ['', 'missing', 'n/a', 'na', 'nan', 'none', 'null', 'other', 'unknown']

--- Processing column: grade ---


/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipykernel_13468/979437001.py:41: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(df[col]):
/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipykernel_13468/979437001.py:73: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if not pd.api.types.is_categorical_dtype(df[col]):


Unique categories after cleaning: 8
grade_was_missing (count): 33
Count 'Unknown': 33, Count 'Other': 0
Top categories (after grouping):
grade
B          663557
C          650053
A          433027
D          324424
E          135639
F           41800
G           12168
Unknown        33
Name: count, dtype: int64

--- Processing column: sub_grade ---


/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipykernel_13468/979437001.py:41: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(df[col]):
/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipykernel_13468/979437001.py:66: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(df[col]):
/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipykernel_13468/979437001.py:70: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df[col] = df[col].replace(rare_labels, "Other")
/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipyk

Unique categories after cleaning: 28
sub_grade_was_missing (count): 33
Count 'Unknown': 33, Count 'Other': 40555
Top categories (after grouping):
sub_grade
C1    145903
B5    140288
B4    139793
B3    131514
C2    131116
C3    129193
C4    127115
B2    126621
Name: count, dtype: int64

--- Processing column: home_ownership ---


...

/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipykernel_13468/979437001.py:41: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(df[col]):
/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipykernel_13468/979437001.py:66: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(df[col]):
/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipykernel_13468/979437001.py:70: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df[col] = df[col].replace(rare_labels, "Other")
/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipyk

Unique categories after cleaning: 12
purpose_was_missing (count): 139473
Count 'Unknown': 139473, Count 'Other': 4224
Top categories (after grouping):
purpose
debt_consolidation    1277877
credit_card            516971
home_improvement       150457
Unknown                139473
major_purchase          50445
medical                 27488
small_business          24689
car                     24013
Name: count, dtype: int64

--- Processing column: addr_state ---


/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipykernel_13468/979437001.py:41: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(df[col]):
/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipykernel_13468/979437001.py:66: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(df[col]):
/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipykernel_13468/979437001.py:70: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df[col] = df[col].replace(rare_labels, "Other")
/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipyk

Unique categories after cleaning: 37
addr_state_was_missing (count): 33
Count 'Unknown': 33, Count 'Other': 98450
Top categories (after grouping):
addr_state
CA       314533
NY       186389
TX       186335
FL       161991
Other     98450
IL        91173
NJ        83132
PA        76939
Name: count, dtype: int64

--- Processing column: application_type ---
Unique categories after cleaning: 3
application_type_was_missing (count): 33
Count 'Unknown': 33, Count 'Other': 0
Top categories (after grouping):
application_type
Individual    2139958
Joint App      120710
Unknown            33
Other               0
Name: count, dtype: int64

Categorical cleanup complete. All processed columns are category dtype and have missing flags.


/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipykernel_13468/979437001.py:41: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(df[col]):
/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipykernel_13468/979437001.py:73: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if not pd.api.types.is_categorical_dtype(df[col]):


In [92]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 75 columns):
 #   Column                        Dtype         
---  ------                        -----         
 0   id                            object        
 1   loan_amnt                     float64       
 2   funded_amnt                   float64       
 3   int_rate                      float64       
 4   installment                   float64       
 5   grade                         category      
 6   sub_grade                     category      
 7   home_ownership                category      
 8   annual_inc                    float64       
 9   verification_status           category      
 10  issue_d                       datetime64[ns]
 11  loan_status                   category      
 12  purpose                       category      
 13  addr_state                    category      
 14  dti                           float64       
 15  delinq_2yrs                   In

In [93]:
# === Step: Verify and Drop datetime columns ===
datetime_cols = ["issue_d", "earliest_cr_line"]

print("=== DATETIME COLUMN CHECK ===")
for col in datetime_cols:
    if col in df.columns:
        print(f"\nColumn: {col}")
        print(f"Dtype: {df[col].dtype}")
        print(f"Missing (NaN): {df[col].isna().sum()} ({100 * df[col].isna().mean():.3f}%)")
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            print("✓ Valid datetime64 dtype")
        else:
            print("⚠️ Not datetime64 — please verify before dropping.")
    else:
        print(f"⚠️ Column {col} not found in DataFrame")

=== DATETIME COLUMN CHECK ===

Column: issue_d
Dtype: datetime64[ns]
Missing (NaN): 33 (0.001%)
✓ Valid datetime64 dtype

Column: earliest_cr_line
Dtype: datetime64[ns]
Missing (NaN): 62 (0.003%)
✓ Valid datetime64 dtype


In [94]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 75 columns):
 #   Column                        Dtype         
---  ------                        -----         
 0   id                            object        
 1   loan_amnt                     float64       
 2   funded_amnt                   float64       
 3   int_rate                      float64       
 4   installment                   float64       
 5   grade                         category      
 6   sub_grade                     category      
 7   home_ownership                category      
 8   annual_inc                    float64       
 9   verification_status           category      
 10  issue_d                       datetime64[ns]
 11  loan_status                   category      
 12  purpose                       category      
 13  addr_state                    category      
 14  dti                           float64       
 15  delinq_2yrs                   In

In [95]:
# Step 1: Identify all _was_ columns
flag_cols = [c for c in df.columns if '_was_' in c]

# Step 2: Show dtype + unique non-null values for each
for col in flag_cols:
    vals = df[col].dropna().unique()
    print(f"{col:30} -> {df[col].dtype} | Unique non-null values: {vals[:10]} | NaNs: {df[col].isna().sum()}")


annual_inc_was_sentinel        -> bool | Unique non-null values: [False  True] | NaNs: 0
annual_inc_was_missing         -> int64 | Unique non-null values: [0 1] | NaNs: 0
annual_inc_was_extreme         -> int8 | Unique non-null values: [0 1] | NaNs: 0
dti_was_sentinel               -> int64 | Unique non-null values: [0 1] | NaNs: 0
dti_was_missing                -> int64 | Unique non-null values: [0 1] | NaNs: 0
dti_was_extreme                -> int64 | Unique non-null values: [0 1] | NaNs: 0
revol_util_was_sentinel        -> bool | Unique non-null values: [False  True] | NaNs: 0
revol_util_was_missing         -> bool | Unique non-null values: [False  True] | NaNs: 0
revol_util_was_extreme         -> bool | Unique non-null values: [False  True] | NaNs: 0
int_rate_was_missing           -> int8 | Unique non-null values: [0 1] | NaNs: 0
int_rate_was_extreme           -> int8 | Unique non-null values: [0 1] | NaNs: 0
installment_was_missing        -> int8 | Unique non-null values: [0 1] | 

In [96]:
# Step 1️⃣ — Verify flag consistency and unify dtypes

# 1. Identify all flag columns
flag_cols = [c for c in df.columns if '_was_' in c]

# 2. Convert dtype inconsistencies
for col in flag_cols:
    if df[col].dtype == 'bool':
        df[col] = df[col].astype('int8')   # True/False → 1/0
    elif df[col].dtype == 'int64':
        df[col] = df[col].astype('int8')   # Downcast int64 → int8
    elif str(df[col].dtype) == 'Int64':    # Handle pandas nullable int
        df[col] = df[col].fillna(0).astype('int8')

# 3. Confirm everything is clean and consistent
print("\n=== FLAG COLUMN TYPE CHECK ===")
print(df[flag_cols].dtypes.value_counts())

print("\nSample unique values (should be [0 1]):")
for col in flag_cols[:10]:
    print(f"{col:30} -> {df[col].unique()}")



=== FLAG COLUMN TYPE CHECK ===
int8    45
Name: count, dtype: int64

Sample unique values (should be [0 1]):
annual_inc_was_sentinel        -> [0 1]
annual_inc_was_missing         -> [0 1]
annual_inc_was_extreme         -> [0 1]
dti_was_sentinel               -> [0 1]
dti_was_missing                -> [0 1]
dti_was_extreme                -> [0 1]
revol_util_was_sentinel        -> [0 1]
revol_util_was_missing         -> [0 1]
revol_util_was_extreme         -> [0 1]
int_rate_was_missing           -> [0 1]


In [97]:
# === STEP 2️⃣ : VERIFY & CLEAN CATEGORICAL COLUMNS ===
import pandas as pd

# 1️⃣ Identify all categorical columns
cat_cols = df.select_dtypes(include="category").columns.tolist()
print(f"Found {len(cat_cols)} categorical columns: {', '.join(cat_cols)}\n")

# 2️⃣ Clean each categorical column
for col in cat_cols:
    print(f"--- Processing categorical column: {col} ---")
    
    # Drop unused categories (from rare-label grouping or replacements)
    df[col] = df[col].cat.remove_unused_categories()
    
    # Quick health check
    n_unique = df[col].nunique()
    n_missing = df[col].isna().sum()
    categories = list(df[col].cat.categories)
    
    print(f"✓ Unique categories: {n_unique}")
    print(f"✓ Missing values: {n_missing}")
    print(f"✓ Example categories: {categories[:10]}{'...' if len(categories) > 10 else ''}")
    print()

# 3️⃣ Final verification
print("=== CATEGORY CLEANUP SUMMARY ===")
print(df.select_dtypes("category").nunique().sort_values(ascending=False))
print("\nAll categorical columns are cleaned and memory-efficient.")


Found 8 categorical columns: grade, sub_grade, home_ownership, verification_status, loan_status, purpose, addr_state, application_type

--- Processing categorical column: grade ---
✓ Unique categories: 8
✓ Missing values: 0
✓ Example categories: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'Unknown']

--- Processing categorical column: sub_grade ---
✓ Unique categories: 28
✓ Missing values: 0
✓ Example categories: ['A1', 'A2', 'A3', 'A4', 'A5', 'B1', 'B2', 'B3', 'B4', 'B5']...

--- Processing categorical column: home_ownership ---
✓ Unique categories: 5
✓ Missing values: 0
✓ Example categories: ['MORTGAGE', 'OWN', 'RENT', 'Unknown', 'Other']

--- Processing categorical column: verification_status ---
✓ Unique categories: 3
✓ Missing values: 33
✓ Example categories: ['Not Verified', 'Source Verified', 'Verified']

--- Processing categorical column: loan_status ---
✓ Unique categories: 6
✓ Missing values: 0
✓ Example categories: ['Charged Off', 'Current', 'Fully Paid', 'Late (31-120 days)', 'Unkn

In [98]:
# === STEP 4️⃣ : CHECK NUMERIC CONSISTENCY ===
import numpy as np

num_cols = df.select_dtypes(include=["number"]).columns.tolist()
print(f"Found {len(num_cols)} numeric columns.")

for col in num_cols:
    non_numeric = df[col][~df[col].apply(lambda x: isinstance(x, (int, float, np.integer, np.floating, type(pd.NA))))].count()
    if non_numeric > 0:
        print(f"⚠️ {col} has {non_numeric} non-numeric values.")
    else:
        pass

print("\n✅ Numeric columns appear clean if no warnings above.")


Found 64 numeric columns.

✅ Numeric columns appear clean if no warnings above.


In [99]:
df.drop(columns=["issue_d", "earliest_cr_line"], inplace=True)


In [100]:
print("=== FINAL SANITY CHECK ===")
print(df.info(memory_usage='deep'))
print("\nMissing values by dtype:")
print(df.isna().sum().sort_values(ascending=False).head(10))


=== FINAL SANITY CHECK ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 73 columns):
 #   Column                        Dtype   
---  ------                        -----   
 0   id                            object  
 1   loan_amnt                     float64 
 2   funded_amnt                   float64 
 3   int_rate                      float64 
 4   installment                   float64 
 5   grade                         category
 6   sub_grade                     category
 7   home_ownership                category
 8   annual_inc                    float64 
 9   verification_status           category
 10  loan_status                   category
 11  purpose                       category
 12  addr_state                    category
 13  dti                           float64 
 14  delinq_2yrs                   Int64   
 15  fico_range_low                Int64   
 16  fico_range_high               Int64   
 17  inq_last_6mths     

In [101]:
import os
import pandas as pd

# === Set your save path ===
OUTPUT_PARQUET = "/Users/abhinavsaxena/Documents/Project/1/clean_data/clean_pd_dataset.parquet"

# Ensure directory exists
os.makedirs(os.path.dirname(OUTPUT_PARQUET), exist_ok=True)

# --- Save to Parquet ---
print(f"Saving cleaned dataset to:\n{OUTPUT_PARQUET}")
df.to_parquet(OUTPUT_PARQUET, index=False, engine="pyarrow")
print("✅ Save complete.\n")

# --- Verification step (reload to confirm integrity) ---
print("Verifying parquet integrity...")
df_check = pd.read_parquet(OUTPUT_PARQUET, engine="pyarrow")

print("\nShape check (original vs reloaded):", df.shape, "->", df_check.shape)
print("\nDtype consistency check:")
print(df_check.dtypes.value_counts())

print("\n✅ Parquet verification successful — dtypes preserved.")


Saving cleaned dataset to:
/Users/abhinavsaxena/Documents/Project/1/clean_data/clean_pd_dataset.parquet
✅ Save complete.

Verifying parquet integrity...

Shape check (original vs reloaded): (2260701, 73) -> (2260701, 73)

Dtype consistency check:
int8        45
Int64       11
float64      8
object       1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64

✅ Parquet verification successful — dtypes preserved.
